# L41 · 加窗（windowing） FFT 完整流程 — 信号 → 加窗 → FFT → 幅度谱（magnitude spectrum），一条管线跑通

**目标**：理解信号截断导致的频谱泄漏，掌握 Hann 窗的压制原理，完成 `windowed_fft` 并用它观察真实旁瓣（sidelobe）差异。

🔗 **Aurora 连接**：`aurora.audio.stft()` 在每帧调用加窗 FFT；`aurora/audio/windows.py` 提供 `hann`、`hamming` 等窗函数（window function），正是本节要接入的模块。

← **上一课**　[L40 · 频谱分析实战](L40_spectrum.ipynb)

> 上节课学习了 **频谱分析实战**：幅度谱 / 相位谱 / 频率分辨率，440Hz+880Hz 混合信号。  
> 本课将探讨 **加窗 FFT 完整流程**。

## 本课剧情：为什么加窗？——打开收音机比截断收音机好听

想象你在听一段音乐，突然"咔"地一声剪断——这个边缘的跳跃在频率上相当于混入了一阵宽频噪音。这就是**频谱泄漏（spectral leakage）**。

FFT 处理的不是整段信号，而是一帧（frame）。截断一帧相当于把信号乘以矩形窗（rectangular window）：边界处骤降到零，产生频域混叠。时域乘法 → 频域卷积，矩形窗的 sinc 旁瓣把能量"泄漏"到相邻频率桶。

**解法**：用一个在边界处平滑收敛到零的窗函数替换矩形窗——Hann 窗是最常用的选择：

$$w[n] = 0.5\left(1 - \cos\!\frac{2\pi n}{N}\right), \quad n = 0,\ldots,N-1$$

中间保持原信号，两端渐变到零，就像慢慢打开再关上收音机，不再有突兀的跳跃。

**完整流程**（三步管线）：

| 步骤 | 操作 | 效果 |
|---|---|---|
| 1 | `frame = x[start:start+N]` | 取帧 |
| 2 | `frame = frame * w` | 乘以窗函数 |
| 3 | `X = fft(frame)` | FFT → 复数频谱 |

本节任务：实现 `windowed_fft(x, window_type="hann")` — 一行胶合三步管线。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.windows import hann, hamming

## 1. 频谱泄漏

截断信号 = 原始信号 × 矩形窗。时域相乘 ↔ 频域卷积，所以频谱 = 原始频谱 ∗ sinc。sinc 的旁瓣从主瓣向两侧延伸，把本应集中在一个 bin 的能量分散到所有 bin，这就是**频谱泄漏（spectral leakage）**。

矩形窗的旁瓣峰值比主瓣仅低约 `-13 dB`，意味着一个强信号的旁瓣足以淹没旁边的弱信号。

In [ ]:
# 演示：一个恰好不在整数 bin 的正弦波的频谱泄漏
N = 256
fs = 1000.0
# 440 Hz 不是 fs/N 的整数倍 → 频率不对齐 bin
t = np.arange(N) / fs
x = np.sin(2 * np.pi * 440 * t)

X_rect = np.abs(np.fft.rfft(x))
freqs = np.fft.rfftfreq(N, 1/fs)

plt.figure(figsize=(10, 3))
plt.plot(freqs, 20 * np.log10(X_rect + 1e-12))
plt.axvline(440, color='r', linestyle='--', label='440 Hz')
plt.xlabel('频率 (Hz)')
plt.ylabel('幅度 (dB)')
plt.title('矩形窗 FFT — 明显的旁瓣泄漏')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Hann 窗

Hann 窗定义为：

```
w[n] = 0.5 * (1 - cos(2*pi*n / N)),   n = 0, 1, ..., N-1  # 周期型（periodic）
```

左端 w[0]=0，右端 w[N-1]≠0（周期型，与对称型 N-1 分母不同），中间平滑隆起。它的频域响应：
- 主瓣宽度是矩形窗的 **2 倍**（分辨率略降）
- 旁瓣峰值比主瓣低约 **-31 dB**（矩形窗仅 -13 dB），大幅压制泄漏

`aurora/audio/windows.py` 中的 `hann(N)` 返回长度 N 的 Hann 窗向量。

In [ ]:
# 对比三种窗函数的时域形状
N = 64
n = np.arange(N)
w_rect = np.ones(N)
w_hann = hann(N)
w_hamm = hamming(N)

plt.figure(figsize=(10, 3))
plt.plot(n, w_rect, label='矩形窗')
plt.plot(n, w_hann, label='Hann 窗')
plt.plot(n, w_hamm, label='Hamming 窗', linestyle='--')
plt.xlabel('样本 n')
plt.ylabel('权重')
plt.title('三种窗函数时域形状（N=64）')
plt.legend()
plt.tight_layout()
plt.show()

## 3. 加窗 FFT 流程

标准流程分三步：

```
1. frame     = x_segment          # 从信号中取一帧（长度 N）
2. x_win     = frame * window      # 逐点乘窗函数（同长）
3. spectrum  = FFT(x_win)          # 对加窗帧做 FFT，取模得幅度谱
```

在 STFT 中，对每一帧重复这三步，帧与帧之间有 hop_length 的偏移量。`aurora.audio.stft()` 内部就是这个循环。

In [ ]:
# 展示加窗后信号边界平滑
N = 128
t = np.arange(N) / 1000.0
x_seg = np.sin(2 * np.pi * 440 * t)   # 不对齐的帧
w = hann(N)
x_win = x_seg * w

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(x_seg, label='原始帧')
axes[0].set_title('截断帧（边界有跳变）')
axes[0].set_xlabel('样本')
axes[1].plot(x_win, label='加 Hann 窗后', color='orange')
axes[1].set_title('Hann 加窗后（边界平滑）')
axes[1].set_xlabel('样本')
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `windowed_fft(x, window_type="hann")`

**核心任务**：根据 `window_type` 选择窗函数，乘以输入帧，再做 FFT：

| 步骤 | 代码思路 | 说明 |
|---|---|---|
| 1 | `if window_type == "hann": w = hann(N)` | 从 `aurora.audio.windows` 导入 |
|   | `elif window_type == "hamming": w = hamming(N)` | |
|   | `else: w = np.ones(N)` | "rectangular" 或其他 → 不改变信号 |
| 2 | `return np.fft.fft(x * w)` | 可用 `np.fft.fft` 或 `aurora_fft` |

**验收标准**：
- `window_type="rectangular"` → 结果与 `np.fft.fft(x)` 完全一致（因为 `np.ones(N) * x = x`）
- `window_type="hann"` → 输出形状 `(N,)` 复数数组
- 旁瓣高度：Hann 比矩形窗低约 30 dB

In [ ]:
def windowed_fft(x: np.ndarray, window_type: str = "hann") -> np.ndarray:
    """对输入帧 x 应用指定窗函数后执行 FFT。

    参数
    ----
    x           : 输入帧，形状 (N,)
    window_type : 窗类型，"hann" | "hamming" | "rectangular"

    返回
    ----
    复数 FFT 输出，形状 (N,)
    """
    N = len(x)
    # ✏️ TODO: 根据 window_type 生成窗向量（hann / hamming / ones）
    # 提示：未知类型应 raise ValueError(f"未知窗类型: {window_type}")
    raise NotImplementedError("请实现 windowed_fft")


In [ ]:
# 检查：矩形窗不改变 x，结果与直接 FFT 完全一致
try:
    _out = windowed_fft(np.ones(64), 'rectangular')
    assert np.allclose(_out, np.fft.fft(np.ones(64))), '矩形窗 FFT 应与 np.fft.fft 相等'
    print('✅ windowed_fft(rectangular) 通过')

    x_test = np.random.randn(128)
    out = windowed_fft(x_test, 'hann')
    assert out.shape == (128,), f'输出形状错误：{out.shape}'
    print('✅ windowed_fft(hann) 输出形状正确')

    w = hann(128)
    assert np.isclose(out[0].real, np.sum(x_test * w)), 'bin 0 应等于加窗信号之和'
    print('✅ bin 0 = sum(x * window) 验证通过')

    # Major fix: 补充 hamming 路径测试
    out_h = windowed_fft(x_test, 'hamming')
    assert out_h.shape == (128,), f'hamming 输出形状错误：{out_h.shape}'
    print('✅ windowed_fft(hamming) 输出形状正确')

    # Major fix: 测试未知 window_type 应抛出 ValueError
    try:
        windowed_fft(x_test, 'invalid_window')
        print('❌ 应对未知 window_type 抛出 ValueError，但未抛出')
    except ValueError as e:
        print(f'✅ 未知 window_type 正确抛出 ValueError: {e}')
    except NotImplementedError:
        print('⬜ 请先完整实现 windowed_fft（包括 ValueError 分支），再运行此格')

except NotImplementedError:
    print('⬜ 请先实现 windowed_fft，再运行此格')
except TypeError:
    print('⬜ windowed_fft 返回了非数组类型，请完成 TODO')


## 5. 参数实验：旁瓣对比

**实验设置**：440 Hz 正弦波，采样率（sample rate，sr） 1000 Hz，帧长 256 点。440 Hz 不是 bin 频率分辨率（1000/256 ≈ 3.9 Hz）的整数倍，频率不对齐 bin，泄漏最严重。

**预期现象**：
- `"rectangular"`：主瓣两侧旁瓣高，第一旁瓣比主瓣仅低约 13 dB
- `"hann"`：旁瓣压制明显，第一旁瓣比主瓣低约 31 dB，两条曲线在主瓣附近差距肉眼可见

调整 `freq_hz` 到恰好对齐某个 bin（如 500 Hz = 128 个整周期）后，两种窗的频谱都会退化成单根线，泄漏消失。

In [ ]:
# 参数实验：对比 rectangular vs hann 旁瓣
freq_hz = 440.0   # 修改此值观察对齐/非对齐的差异
fs = 1000.0
N = 256

t = np.arange(N) / fs
x = np.sin(2 * np.pi * freq_hz * t)

X_rect = np.abs(windowed_fft(x, window_type="rectangular")[:N//2])
X_hann = np.abs(windowed_fft(x, window_type="hann")[:N//2])
freqs = np.fft.rfftfreq(N, 1/fs)[:-1]   # N//2 点

# 归一化：主瓣峰值设为 0 dB
X_rect_db = 20 * np.log10(X_rect / X_rect.max() + 1e-12)
X_hann_db = 20 * np.log10(X_hann / X_hann.max() + 1e-12)

# 计算旁瓣差（排除主瓣附近 ±5 bin）
peak_bin = np.argmax(X_rect)
mask = np.ones(len(X_rect), dtype=bool)
mask[max(0, peak_bin-5):peak_bin+6] = False
sidelobe_rect = X_rect_db[mask].max()
sidelobe_hann = X_hann_db[mask].max()
print(f'矩形窗最大旁瓣: {sidelobe_rect:.1f} dB')
print(f'Hann 窗最大旁瓣:  {sidelobe_hann:.1f} dB')
print(f'旁瓣压制差: {sidelobe_rect - sidelobe_hann:.1f} dB')

plt.figure(figsize=(11, 4))
plt.plot(freqs, X_rect_db, label='矩形窗', alpha=0.8)
plt.plot(freqs, X_hann_db, label='Hann 窗', alpha=0.8)
plt.axhline(-13, color='gray', linestyle=':', linewidth=0.8, label='-13 dB 参考')
plt.axhline(-31, color='brown', linestyle=':', linewidth=0.8, label='-31 dB 参考')
plt.ylim(-80, 5)
plt.xlabel('频率 (Hz)')
plt.ylabel('归一化幅度 (dB)')
plt.title(f'频谱泄漏对比（{freq_hz} Hz，fs={fs} Hz，N={N}）')
plt.legend()
plt.tight_layout()
plt.show()

## 本课收束

本节实现了 `windowed_fft(x, window_type)`，它在加窗后输出长度为 N 的复数 FFT 频谱。Hann 窗将旁瓣压低约 18 dB（旁瓣峰值 −31 dB vs 矩形窗 −13 dB）的能力来自 `aurora/audio/windows.py` 中 `hann(N)` 的余弦权重设计。下一课（L42 / FFT 图形化）将通过蝴蝶图和频谱对比，直观展示 FFT 的计算结构与纯音、和弦、噪声的频谱形态。

## ✏️ 白板挑战：加窗 FFT 手算与推理（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：频谱泄漏（spectral leakage）的根本原因是什么？  
（提示：时域截断 = 乘以矩形窗 → 频域做什么运算？）

**问 2**：Hann 窗 `w[n] = 0.5(1 - cos(2πn/N))` 的边界值是多少？  
（n=0 时 w[0]=?，n=N-1 时 w[N-1]=?）

**问 3**：`windowed_fft([1, 1, 1, ..., 1], window_type="rectangular")` 的结果是什么？  
（全为1的信号只有DC分量，`X[0]=N`，其余`X[k]=0`）

**问 4**：如果 `window_type="hann"` 的旁瓣比矩形窗低约 31 dB，用 dB 表示等于多少倍振幅？  
（提示：20·log₁₀(ratio) = -31 dB → ratio = 10^(-31/20) ≈ ?）

**问 5**：为什么在 STFT（短时傅里叶变换）中，相邻帧通常要有 50% 重叠（overlap）？  
（提示：Hann 窗中间最大，两端接近零——窗函数抑制了哪里的信息？）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：频谱泄漏原因 — 时域乘以矩形窗 ↔ 频域卷积 sinc
# 验证：矩形窗的 DTFT 是 sinc 函数
N = 16
rect = np.ones(N)
X_rect = np.fft.fft(rect, n=256)  # 零填充放大频域
# sinc 旁瓣存在
assert np.max(np.abs(X_rect[8:128])) > 0.1  # 旁瓣非零
print(f"Q1 ✅  时域截断=乘以矩形窗，频域=与sinc卷积，sinc旁瓣把能量扩散到邻近bin")

# 问2：Hann 窗边界值
N_h = 8
n_arr = np.arange(N_h)
w_hann = 0.5 * (1 - np.cos(2 * np.pi * n_arr / N_h))
assert np.isclose(w_hann[0], 0.0, atol=1e-12)
assert np.isclose(w_hann[-1], 1 - np.cos(2 * np.pi * (N_h-1) / N_h)) * 0.5  # 接近零
w0 = w_hann[0]
wNm1 = w_hann[-1]
print(f"Q2 ✅  w[0]={w0:.6f}（=0），w[N-1]={wNm1:.6f}（≈0，但非精确0，N点窗）")

# 问3：矩形窗全1信号 → DC分量=N
N3 = 16
x_ones = np.ones(N3)
X3 = np.fft.fft(x_ones)
assert np.isclose(X3[0], N3 + 0j, atol=1e-12)
assert np.allclose(X3[1:], 0+0j, atol=1e-12)
try:
    X3_mine = windowed_fft(x_ones, "rectangular")
    assert np.allclose(X3_mine, X3, atol=1e-9), f"rectangular 应与 np.fft.fft 一致"
    print(f"Q3 ✅  windowed_fft([1..1], 'rectangular')[0]={X3_mine[0]:.1f}，其余≈0")
except NotImplementedError:
    print(f"Q3 ✅  全1信号 FFT: X[0]={X3[0]:.1f}=N，X[k≠0]=0（纯DC）（windowed_fft 待实现）")

# 问4：-31 dB → 振幅比
db_diff = -31.0
ratio = 10 ** (db_diff / 20)
assert abs(ratio - 0.0282) < 0.01  # ≈ 0.028
print(f"Q4 ✅  -31 dB → 振幅比 = 10^(-31/20) ≈ {ratio:.4f}（约为矩形窗旁瓣的 1/35）")

# 问5：50% 重叠原因 — Hann 窗补偿两端信息损失
# 验证 Hann 窗 50% 重叠时重建系数接近常数
N5 = 8
w5 = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N5) / N5))
hop = N5 // 2
# OLA (overlap-add) 重建：w[n] + w[n + hop] 对 n=0..hop-1
ola_sum = w5[:hop] + w5[hop:]
# 理想情况下应近似为常数（Hann窗50%重叠满足COLA条件）
assert np.std(ola_sum) < 0.05 * np.mean(ola_sum), f"50%重叠OLA不平稳：{ola_sum}"
print(f"Q5 ✅  50%重叠时 Hann 窗 OLA 和≈{np.mean(ola_sum):.4f}（平稳），两端信息不丢失")
print("\n🎉 加窗 FFT 白板挑战通过！windowed_fft = frame × window → fft。")

In [ ]:
# ✏️ 本课自评
l41_review = {
    "spectral_leakage_cause":  None,  # 记住泄漏=矩形窗时域截断→频域sinc卷积？True/False
    "hann_window_formula":     None,  # 记住 w[n]=0.5(1-cos(2πn/N))，边界=0？True/False
    "windowed_fft_impl":       None,  # windowed_fft 实现并通过 rectangular 断言？True/False
    "sidelobe_reduction":      None,  # 理解 Hann 窗将旁瓣压低约 31 dB？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l41_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l41_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L41 全部通关！进入 L42：FFT 图形化（可视化课）')

---

→ **下一课**　[L42 · FFT 图形化](L42_visual_fft.ipynb)

> 下节课将学习 **FFT 图形化**：蝴蝶图 + 纯音 / 和弦 / 噪声的频谱形态对比。